# Simple soil CNN training (Google Colab)

This beginner-friendly notebook trains a 7-class soil-photo classifier with MobileNetV2 transfer learning. It expects one folder per soil class. It removes exact duplicate image files before splitting the data into train, validation, and test groups.

**Important:** the earlier experiment scored 84.2% on its own held-out split of 171 images. This notebook may get a different score because the random split, training run, and Colab environment can differ. A photo classifier does not measure soil nutrients or fertility.


**Saved notebook outputs:** the code cells below were executed once on the project's original soil-image folders and their outputs are saved under the cells. Colab may show different scores because its training run and hardware can differ.


## Additional Colab notebooks

- [Colab notebook 1](https://colab.research.google.com/drive/14TdMqXOTOEoXos09ZV-1VpFxVaFbSl7N?usp=sharing)
- [Colab notebook 2](https://colab.research.google.com/drive/1hn6uXGly-c6IUHceK_IBQ_k4j5DZOAQP?usp=sharing)


## 1. Choose the Colab runtime and dataset folder

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. Download and unzip the dataset so the folder has this structure:

```
/content/soil_images/Alluvial_Soil/photo1.jpg
/content/soil_images/Arid_Soil/photo2.jpg
...
```

Change only `DATA_DIR` below if your folder has a different name. The class folders should be directly inside that folder.


In [1]:
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("/content/soil_images")  # Change this one line if needed.
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.16.2
GPU devices: []


In [2]:
# Check that the dataset folder contains one subfolder for each soil class.
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR}. Unzip your dataset in Colab and update DATA_DIR above."
    )

class_names = sorted(folder.name for folder in DATA_DIR.iterdir() if folder.is_dir())
print("Classes:", class_names)
if len(class_names) < 2:
    raise ValueError("I need at least two class folders inside DATA_DIR.")


Classes: ['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']


## 2. Remove exact duplicates and make the data splits

The same exact image should not appear in both training and test data. We hash the image files and keep only the first copy. Then we split images by class: 70% for training, 15% for validation, and 15% for the final test.


In [3]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
seen_hashes = set()
file_paths = []
labels = []
duplicate_count = 0

for class_id, class_name in enumerate(class_names):
    class_folder = DATA_DIR / class_name
    for path in sorted(class_folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in image_extensions:
            continue
        image_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        if image_hash in seen_hashes:
            duplicate_count += 1
            continue
        seen_hashes.add(image_hash)
        file_paths.append(str(path))
        labels.append(class_id)

file_paths = np.array(file_paths)
labels = np.array(labels, dtype=np.int32)
print("Unique images:", len(file_paths), "| exact duplicates removed:", duplicate_count)

# 70% train, 15% validation, 15% test; stratify preserves class balance.
train_paths, other_paths, train_y, other_y = train_test_split(
    file_paths, labels, test_size=0.30, random_state=SEED, stratify=labels
)
valid_paths, test_paths, valid_y, test_y = train_test_split(
    other_paths, other_y, test_size=0.50, random_state=SEED, stratify=other_y
)
print("Train / validation / test:", len(train_paths), len(valid_paths), len(test_paths))


Unique images: 1140 | exact duplicates removed: 49
Train / validation / test: 798 171 171


## 3. Load and prepare image batches

TensorFlow reads each image, resizes it to 224 × 224, and groups it into batches. The model itself scales pixel values for MobileNetV2.


In [4]:
def load_image(path, label):
    # Pillow handles JPG, PNG, and WEBP files consistently.
    def open_with_pillow(path_bytes):
        file_path = path_bytes.decode("utf-8")
        with Image.open(file_path) as image:
            return np.asarray(image.convert("RGB"), dtype=np.float32)

    image = tf.numpy_function(open_with_pillow, [path], tf.float32)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, label


def make_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = make_dataset(train_paths, train_y, training=True)
valid_ds = make_dataset(valid_paths, valid_y)
test_ds = make_dataset(test_paths, test_y)

# Give less-common classes a little extra weight during training.
weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_y), y=train_y
)
class_weights = {int(class_id): float(weight) for class_id, weight in zip(np.unique(train_y), weights)}
print("Class weights:", class_weights)


Class weights: {0: 3.257142857142857, 1: 0.5846153846153846, 2: 0.6951219512195121, 3: 0.76, 4: 0.8507462686567164, 5: 1.52, 6: 2.533333333333333}


## 4. Build the image model

MobileNetV2 already learned useful visual patterns from ImageNet. First we freeze its learned layers and train a small new classification head for our soil classes.


In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "functional_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)      

## 5. Training stage one: train the new classification head

We keep the original MobileNetV2 layers frozen. Early stopping stops when validation accuracy stops improving, which helps reduce overfitting.


In [6]:
stage1_path = "/content/soil_stage1_best.keras"
stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(stage1_path, monitor="val_accuracy", mode="max", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True),
]

history1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=12,
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=2,
)

stage1_model = tf.keras.models.load_model(stage1_path)
stage1_val_accuracy = stage1_model.evaluate(valid_ds, verbose=0)[1]
print("Stage-one best validation accuracy:", round(float(stage1_val_accuracy), 4))


Epoch 1/12
25/25 - 20s - 800ms/step - accuracy: 0.3622 - loss: 1.7785 - val_accuracy: 0.5556 - val_loss: 1.1869
Epoch 2/12
25/25 - 11s - 423ms/step - accuracy: 0.6491 - loss: 1.0801 - val_accuracy: 0.7310 - val_loss: 0.7817
Epoch 3/12
25/25 - 11s - 425ms/step - accuracy: 0.7193 - loss: 0.8615 - val_accuracy: 0.7602 - val_loss: 0.6721
Epoch 4/12
25/25 - 9s - 374ms/step - accuracy: 0.7719 - loss: 0.7130 - val_accuracy: 0.7661 - val_loss: 0.6582
Epoch 5/12
25/25 - 9s - 353ms/step - accuracy: 0.8083 - loss: 0.6142 - val_accuracy: 0.7953 - val_loss: 0.5702
Epoch 6/12
25/25 - 11s - 460ms/step - accuracy: 0.8446 - loss: 0.5351 - val_accuracy: 0.8012 - val_loss: 0.5569
Epoch 7/12
25/25 - 11s - 442ms/step - accuracy: 0.8484 - loss: 0.4997 - val_accuracy: 0.8246 - val_loss: 0.5387
Epoch 8/12
25/25 - 11s - 452ms/step - accuracy: 0.8546 - loss: 0.4727 - val_accuracy: 0.7953 - val_loss: 0.5458
Epoch 9/12
25/25 - 9s - 370ms/step - accuracy: 0.8697 - loss: 0.4353 - val_accuracy: 0.8304 - val_loss: 0.

## 6. Training stage two: carefully fine-tune MobileNetV2

Now we unfreeze only the last 35 MobileNetV2 layers and train with a much smaller learning rate. This gently adapts some pretrained features to soil photos.


In [7]:
base_model.trainable = True
for layer in base_model.layers[:-35]:
    layer.trainable = False
# Keep Batch Normalization stable when the training set is small.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

stage2_path = "/content/soil_stage2_best.keras"
stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(stage2_path, monitor="val_accuracy", mode="max", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
]

history2 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=stage2_callbacks,
    verbose=2,
)

stage2_model = tf.keras.models.load_model(stage2_path)
stage2_val_accuracy = stage2_model.evaluate(valid_ds, verbose=0)[1]
print("Stage-two best validation accuracy:", round(float(stage2_val_accuracy), 4))


Epoch 1/10
25/25 - 21s - 826ms/step - accuracy: 0.8885 - loss: 0.3701 - val_accuracy: 0.8480 - val_loss: 0.4961 - learning_rate: 1.0000e-05
Epoch 2/10
25/25 - 15s - 586ms/step - accuracy: 0.8860 - loss: 0.3256 - val_accuracy: 0.8655 - val_loss: 0.4674 - learning_rate: 1.0000e-05
Epoch 3/10
25/25 - 13s - 530ms/step - accuracy: 0.8997 - loss: 0.2899 - val_accuracy: 0.8480 - val_loss: 0.4710 - learning_rate: 1.0000e-05
Epoch 4/10
25/25 - 12s - 476ms/step - accuracy: 0.9110 - loss: 0.2668 - val_accuracy: 0.8538 - val_loss: 0.4625 - learning_rate: 1.0000e-05
Epoch 5/10
25/25 - 13s - 502ms/step - accuracy: 0.9085 - loss: 0.2581 - val_accuracy: 0.8480 - val_loss: 0.4746 - learning_rate: 1.0000e-05
Epoch 6/10
25/25 - 12s - 497ms/step - accuracy: 0.9148 - loss: 0.2571 - val_accuracy: 0.8070 - val_loss: 0.4722 - learning_rate: 1.0000e-05
Stage-two best validation accuracy: 0.8655


## 7. Keep the best validation model, then test it once

We select between the two saved models using validation accuracy. The test set is kept aside until now so it remains an honest final check.


In [8]:
if stage2_val_accuracy >= stage1_val_accuracy:
    best_model = stage2_model
    print("Selected fine-tuned model.")
else:
    best_model = stage1_model
    print("Selected frozen-base model because it validated better.")

OUTPUT_MODEL = "/content/soil_cnn_colab.keras"
best_model.save(OUTPUT_MODEL)

probabilities = best_model.predict(test_ds, verbose=0)
predicted_y = np.argmax(probabilities, axis=1)
print("Test accuracy:", round(accuracy_score(test_y, predicted_y), 4))
print("Test macro F1:", round(f1_score(test_y, predicted_y, average="macro"), 4))
print(classification_report(test_y, predicted_y, target_names=class_names, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(test_y, predicted_y))


Selected fine-tuned model.
Test accuracy: 0.807
Test macro F1: 0.7737
               precision    recall  f1-score   support

Alluvial_Soil       0.56      0.62      0.59         8
    Arid_Soil       0.78      0.83      0.80        42
   Black_Soil       0.97      0.91      0.94        35
Laterite_Soil       0.76      0.69      0.72        32
Mountain_Soil       0.86      0.89      0.88        28
     Red_Soil       0.67      0.75      0.71        16
  Yellow_Soil       0.88      0.70      0.78        10

     accuracy                           0.81       171
    macro avg       0.78      0.77      0.77       171
 weighted avg       0.81      0.81      0.81       171

Confusion matrix (rows = actual, columns = predicted):
[[ 5  1  0  2  0  0  0]
 [ 2 35  0  1  3  1  0]
 [ 0  2 32  1  0  0  0]
 [ 1  4  0 22  0  4  1]
 [ 0  2  0  0 25  1  0]
 [ 1  0  1  2  0 12  0]
 [ 0  1  0  1  1  0  7]]


## 8. Download the trained model from Colab

Run this final cell to download the model file. Keep the class-name list in the same sorted order when using the model in another app.


In [9]:
try:
    from google.colab import files
    print("Downloading the saved model...")
    files.download(OUTPUT_MODEL)
except ImportError:
    print("Outside Colab: model saved in the runtime. In Colab, this cell downloads it.")


Outside Colab: model saved in the runtime. In Colab, this cell downloads it.


## Beginner notes

- A different run can produce a different accuracy; do not expect exactly 84.2% every time.
- The test score measures performance on this dataset split, not every farm, camera, season, or region.
- If one class has very few photos, collect more real examples for that class.
- This CNN recognizes visual categories. It does not determine whether soil is fertile; that requires soil measurements or a lab test.
